## Movies Dataset 2024 - Grupo 9
<hr style="border:1px solid gray">

# 1. Configuración inicial

## 1.1 Correr con UV

```shell
uv sync
uv run jupyter lab
```

## 1.2 Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import kagglehub
from scipy import stats as st
from scipy.stats import describe
from pathlib import Path
import shutil

## 1.3 Importart dataset

In [ ]:
# Create data directory if it doesn't exist
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Download dataset (latest version by default)
download_path = kagglehub.dataset_download("asaniczka/tmdb-movies-dataset-2023-930k-movies")

# Copy to local data directory
csv_file = next(Path(download_path).glob("*.csv"))
local_path = data_dir / csv_file.name
shutil.copy2(csv_file, local_path)

print(f"Dataset copied to: {local_path}")

## 1.4 Carga del dataset

In [ ]:
# Cargamos el dataset desde el directorio local de datos
data_path = "data/TMDB_movie_dataset_v11.csv"
movies = pd.read_csv(data_path)

# 2. Descripción del dataset IMDB Movies 2024


| Variable              | Descripción                                                                                  |
|-----------------------|----------------------------------------------------------------------------------------------|
| `id`                  | Unique identifier for each movie. (type: int)                                                |
| `title`               | Title of the movie (usually in English or the display title). (type: str)                    |
| `vote_average`        | Average vote or rating given by viewers (typically 0–10 scale). (type: float)                |
| `vote_count`          | Total count of votes received for the movie. (type: int)                                     |
| `status`              | Current status of the movie (Released, Rumored, Post Production, etc.). (type: str)          |
| `release_date`        | Date when the movie was released (format usually YYYY-MM-DD). (type: str)                    |
| `revenue`             | Total revenue generated by the movie (in USD; 0 often means unknown/missing). (type: int)    |
| `runtime`             | Duration of the movie in minutes (0 usually means unknown). (type: int)                      |
| `adult`               | Indicates if the movie is only for adult audiences (true/false). (type: bool)                |
| `backdrop_path`       | Relative path to the backdrop image for the movie . (type: str)                              |
| `budget`              | Estimated production budget of the movie (USD; 0 often means unknown/missing). (type: int64) |
| `homepage`            | Official website URL of the movie (empty string or null if not available). (type: str)       |
| `imdb_id`             | IMDb identifier for the movie (format: tt followed by 7–8 digits). (type: str)               |
| `original_language`   | ISO 639-1 code of the original language of the movie (e.g., 'en', 'fr', 'es'). (type: str)   |
| `original_title`      | Original title of the movie (in its original language). (type: str)                          |
| `overview`            | Brief plot summary or synopsis of the movie. (type: str)                                     |
| `popularity`          | Popularity score assigned by TMDB (higher = more popular). (type: float64)                   |
| `poster_path`         | Relative path to the poster image for the movie (prepend https://image.tmdb.org) (type: str) |
| `tagline`             | Short marketing tagline or slogan of the movie (often empty). (type: str)                    |
| `genres`              | List of genres (usually as comma-separated string or JSON in some datasets). (type: str)     |
| `production_companies`| Name(s) of the main production company/companies (often comma-separated or JSON) (type: str) |
| `production_countries`| Country/countries where the movie was produced (often comma-separated or JSON). (type: str)  |
| `spoken_languages`    | Language(s) spoken in the movie (comma-separated or JSON. ISO codes and names). (type: str)  |
| `keywords`            | Descriptive keywords/tags linked with the movie (often comma-separated or JSON). (type: str) |

# 3. Primera inspección

## 3.1 Dimensiones del dataset original

In [ ]:
print(f"Dimensiones del dataset original: {movies.shape}")
# exclude columns w/o valuable data (e.g. backdrop_path, imdb_id,..)
columns = ['title', 'vote_average', 'vote_count', 'status', 'release_date', 'revenue', 'runtime', 'adult', 'budget', 'original_language', 'popularity', 'genres', 'production_companies', 'production_countries', 'keywords']
movies[columns].head()

In [ ]:
movies.info()

In [ ]:
movies[columns].describe()

## 3.2 Análisis de las variables numéricas

#### Max/Mins

In [ ]:
# Maximum budget
max_budget = movies['budget'].max()
print(f"Maximum budget: ${max_budget:,}")

# Minimum budget
min_budget = movies['budget'].min()
print(f"Minimum budget: ${min_budget:,}")

# Maximum revenue
max_budget = movies['revenue'].max()
print(f"Maximum revenue: ${max_budget:,}")

# Minimum revenue
min_budget = movies['revenue'].min()
print(f"Minimum revenue: ${min_budget:,}")

# Maximum popularity
max_budget = movies['popularity'].max()
print(f"Maximum popularity: {max_budget:,}")

# Minimum popularity
min_budget = movies['popularity'].min()
print(f"Minimum popularity: {min_budget:,}")

## 3.3 Histogramas

In [ ]:
# format large values for human readab ility
def human_format_K(num, pos=None):
    """
    Format numbers nicely with K.
    
    Examples:
    - human_format_K(4500)          → '4.5K'
    - human_format_K(4500000)       → '4500K'
    """
    if num == 0:
        return "0"
    
    else:
        value = num / 1_000.0
        suffix = 'K'
        # For thousands: usually whole number unless very small
        fmt = ".1f" if value < 10 else ".0f"
    
    return f"{value:{fmt}}{suffix}"

# format to Million
def human_format_M(num, pos=None):
    """
    Format numbers nicely with M.
    
    Examples:
    - human_format_M(4500000) → '4.5M'
    """
    if num == 0:
        return "0"
    
    else:
        # Force million scale
        value = num / 1_000_000.0
        suffix = 'M'
        # Show one decimal if < 10M, otherwise whole number
        fmt = ".1f" if value < 10 else ".0f"
    
    return f"{value:{fmt}}{suffix}"

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# --- Budget ---
valid_budget = movies['budget'].replace(0, np.nan)
ax[0].hist(np.log1p(valid_budget)) # using log to compress large values
ax[0].set_title('Budget')
ax[0].set_xlabel('Budget (USD)')
ax[0].set_ylabel('Number of movies')
ax[0].xaxis.set_major_formatter(ticker.FuncFormatter(human_format_M))

# --- Revenue ---
valid_revenue = movies['revenue'].replace(0, np.nan)
ax[1].hist(np.log1p(valid_revenue)) # using log to compress large values
ax[1].set_title('Revenue')
ax[1].set_xlabel('Revenue (in Million USD)')
ax[1].xaxis.set_major_formatter(ticker.FuncFormatter(human_format_M))

# --- Popularity ---
valid_pop = movies['popularity'].replace(0, np.nan)
ax[2].hist(np.log1p(valid_pop)) # using log to compress large values
ax[2].set_title('Popularity')
ax[2].set_xlabel('Popularity score')
ax[2].set_ylabel('Number of movies')

plt.show()

### Movies per month
<hr style="border:1px solid gray">


Standarize dates:

In [ ]:
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')

Visualize:

In [ ]:
movies['release_month'] = movies['release_date'].dt.month
ax = movies['release_month'].hist(bins=24)
ax.set_title('Movies per month')
plt.show()

### 1. Medidas de tendencia central: media, mediana y moda
<hr style="border:1px solid gray">


#### Media

In [ ]:
np.mean(movies['budget'])    # Budget

In [ ]:
np.mean(movies['revenue']) # Revenue

In [ ]:
np.mean(movies['popularity']) # Popularity

#### Mediana

In [ ]:
np.median(movies['budget'])  # Budget

In [ ]:
np.median(movies['revenue']) # Revenue

In [ ]:
np.median(movies['popularity']) # Popularity

#### Moda

In [ ]:
movies['budget'].mode()[0]     # Budget

In [ ]:
mode_age = st.mode(movies['budget'], keepdims=False)  # con SciPy (no funciona con nulos)
mode_age.mode

In [ ]:
movies['revenue'].mode()[0]     # Revenue

In [ ]:
movies['popularity'].mode()[0]     # Popularity

#### Por qué Moda y Mediana de 0 para budget y revenue?
Hay muchos 0s. IMDB usa el 0 para representar "unknown", no el valor 0.

In [ ]:
(movies['budget'] == 0).sum()

In [ ]:
(movies['revenue'] == 0).sum()

In [ ]:
(movies['budget'] == 0).mean()

In [ ]:
(movies['revenue'] == 0).mean()

In [ ]:
movies['budget'].replace(0, np.nan).mode()[0]     # moda Budget excluyendo 0s

In [ ]:
movies['revenue'].replace(0, np.nan).mode()[0]     # moda Revenue excluyendo 0s

# 4. Análisis de inconsistencias

## 4.1 Películas duplicadas

In [ ]:
# Dataset information
print(f"Amount of movies (rows): {len(movies)}")
print(f"Amount of unique movies: {movies['id'].nunique()}")
print(f"Amount of columns: {len(movies.columns)}")

# Check for duplicate IDs
duplicate_ids = movies['id'].duplicated().sum()
if duplicate_ids > 0:
    print(f"\n Found {duplicate_ids} duplicate movie IDs!")
else:
    print("\nAll movie IDs are unique")

In [ ]:
# Visualizar información del dataset
fig, ax = plt.subplots(figsize=(10, 6))

# Datos
categorias = ['Filas totales', 'IDs únicos', 'Duplicados']
valores = [len(movies), movies['id'].nunique(), len(movies) - movies['id'].nunique()]
colores = ['#3498db', '#2ecc71', '#e74c3c']

# Crear gráfico de barras
bars = ax.bar(categorias, valores, color=colores)

# Configurar etiquetas y título
ax.set_title('Información del Dataset', fontsize=14, fontweight='bold')
ax.set_ylabel('Cantidad', fontsize=12)

# Formatear el eje Y para mostrar valores completos
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# Ajustar escala del eje Y
max_val = max(valores)
ax.set_ylim(0, max_val * 1.1)

# Agregar valores sobre las barras
for bar, valor in zip(bars, valores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + max_val * 0.01,
             f'{valor:,}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Agregar grid para mejor lectura
ax.grid(axis='y', alpha=0.3)

# Rotar etiquetas si es necesario
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()